In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MongoDB_DataLake_Project")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [21]:
import os

from dotenv import load_dotenv
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv(r"C:\Professional_project\MongoDB_DataLake\credentials.env")

ACCOUNT_NAME = os.getenv("AZURE_STORAGE_ACCOUNT")
ACCOUNT_KEY = os.getenv("AZURE_STORAGE_KEY")

ACCOUNT_URL = f"https://{ACCOUNT_NAME}.dfs.core.windows.net"

service_client = DataLakeServiceClient(
    account_url=ACCOUNT_URL,
    credential=ACCOUNT_KEY
)

file_system_client = service_client.get_file_system_client("bronze")
directory_client = file_system_client.get_directory_client("mongodb")

files = list(directory_client.get_paths())

latest_file = max(
    files,
    key=lambda x: x.last_modified
)

print(latest_file.name)

mongodb/transactions_20260705_124610.json


In [24]:
download_path = r"C:\Professional_project\MongoDB_DataLake\temp\bronze\transactions.json"

file_client = file_system_client.get_file_client(latest_file.name)

download = file_client.download_file()

with open(download_path, "wb") as f:
    f.write(download.readall())

print("Download completed!")

Download completed!


In [25]:
df = spark.read.json(download_path)

df.printSchema()

df.show(5, truncate=False)

root
 |-- destination_account: struct (nullable = true)
 |    |-- account_id: string (nullable = true)
 |    |-- balance_after: double (nullable = true)
 |    |-- balance_before: double (nullable = true)
 |-- fraud: struct (nullable = true)
 |    |-- is_flagged: boolean (nullable = true)
 |    |-- is_fraud: boolean (nullable = true)
 |-- origin_account: struct (nullable = true)
 |    |-- account_id: string (nullable = true)
 |    |-- balance_after: double (nullable = true)
 |    |-- balance_before: double (nullable = true)
 |-- step: long (nullable = true)
 |-- transaction: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- type: string (nullable = true)

+------------------------------------+--------------+-------------------------------------+----+---------------------+
|destination_account                 |fraud         |origin_account                       |step|transaction          |
+------------------------------------+--------------+------------------